# MET Atlas — Highlight Tahmini ve Sergi Seçimi (ML)

**Araştırma sorusu:** Bir eserin *highlight* olması, eserin mevcut özelliklerinden
(bölüm, sınıf, kültür, boyut) tahmin edilebilir mi?
Hedef değişken: `is_highlight`.

**Kısaca ne yapıyoruz?**
1. BigQuery'deki `mart_met_objects` tablosundan eserleri çekiyoruz.
2. Boyut metnini (ör. `"51.4 x 64.6 cm"`) sayıya çeviriyoruz.
3. Basit bir **lojistik regresyon** modeli ile her esere "highlight olma olasılığı" veriyoruz.
4. Bu olasılıkları kullanarak, sınırlı alana sahip odalara **en çok highlight beklenen** eserleri
   yerleştiriyoruz (knapsack / sırt çantası problemi).
5. Sonuçları Power BI için CSV olarak indiriyoruz.

> ⚠️ Model bir eserin highlight olup olmayacağını *tahmin* eder; highlight seçiminin
> *nedenini* açıklamaz. "Bu özellik highlight yapar" gibi nedensel bir sonuç çıkarmıyoruz.

## İçindekiler

0. [Nasıl çalıştırılır?](#scrollTo=nasil)
1. [Veriyi çekme](#scrollTo=b1)
2. [Özellik hazırlama ve train/test bölme](#scrollTo=b2)
3. [Model: Lojistik regresyon](#scrollTo=b3)
4. [Olasılıkları gerçek orana çevirme (kalibrasyon)](#scrollTo=b4)
5. [Tek oda knapsack](#scrollTo=b5)
6. [Çoklu oda knapsack (MILP)](#scrollTo=b6)
7. [Duyarlılık analizi](#scrollTo=b7)
8. [Power BI için dışa aktarma](#scrollTo=b8)
9. [Sonuçlar ve sınırlamalar](#scrollTo=b9)
10. [Kısa sözlük](#scrollTo=sozluk)

## 1. Veriyi çekme

`mart_met_objects` tablosundan model için gereken sütunları alıyoruz.

- Sadece **boyut bilgisi olan** eserler (`dimensions IS NOT NULL`).
- **The Libraries** bölümü hariç (neden: hücredeki nota bak).
- `sergileme_tipi`: eserin duvarda mı, vitrinde mi, zeminde mi sergileneceğine dair
  **kaba bir tahmin** (classification metnindeki anahtar kelimelerden). Bu bir varsayım,
  Met'in gerçek sergileme bilgisi değil.

In [1]:
from google.colab import auth
auth.authenticate_user()          # Google hesabınla BigQuery'ye giriş
from google.cloud import bigquery
import pandas as pd
import numpy as np

# Sorgunun faturalandığı Google Cloud projesi (kendi projenle değiştirebilirsin)
client = bigquery.Client(project="modular-robot-509815-n3")

# Notlar:
# - "The Libraries" bölümü çıkarıldı: eserlerin ~%96'sı highlight görünüyor, bu bir veri
#   hatası/istisna; modeli yanıltmasın ("hile" yapmasın) diye dışarıda.
# - is_on_view SADECE knapsack karşılaştırması için çekiliyor, modele ÖZELLİK OLARAK
#   VERİLMİYOR. Sergilenen eserler zaten highlight olmaya yatkın, bu bilgi sızıntı olur.

query = """
SELECT
    object_id,
    object_number,
    is_highlight,
    is_on_view,
    department,
    classification,
    culture,
    dimensions
FROM
    `the-met-open-access.met_data.mart_met_objects`
WHERE
    dimensions IS NOT NULL
    AND department != 'The Libraries'
"""

df = client.query(query).to_dataframe()
df["is_highlight"] = df["is_highlight"].fillna(False).astype(bool)

# Sergileme tipi (classification'dan türetilir, sızıntı yok)
def sergi_tipini_belirle(classification):
    classification = str(classification).lower()

    vitrin_anahtar_kelimeler = ['coin', 'jewelry', 'miniature', 'gems', 'seal', 'amulet']
    if any(kelime in classification for kelime in vitrin_anahtar_kelimeler):
        return 'Vitrin'

    duvar_anahtar_kelimeler = ['painting', 'drawing', 'photograph', 'print', 'textile']
    if any(kelime in classification for kelime in duvar_anahtar_kelimeler):
        return 'Duvar'

    return 'Zemin'

df['sergileme_tipi'] = df['classification'].apply(sergi_tipini_belirle)

print("Veri Başarıyla Çekildi!")
print(f"Toplam Eser Sayısı: {df.shape[0]}")
print("\nSergileme Tiplerine Göre Eser Dağılımı:")
print(df['sergileme_tipi'].value_counts())
print(f"\nGenel Highlight Oranı: {df['is_highlight'].mean():.4f}")


Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/modular-robot-509815-n3/jobs?prettyPrint=false: Access Denied: Project modular-robot-509815-n3: User does not have bigquery.jobs.create permission in project modular-robot-509815-n3.

Location: None
Job ID: 33af43ff-d3bd-4421-a7bb-dae4bacfb5d8


### 1.1 Kontrol: Boyutu boş olan eserleri dışarıda bırakmak sonucu bozar mı?

Boyutu olmayan eserlerin highlight oranı (%0,18) boyutu olanlardan (%0,52) düşük.
Yani filtre, veri setinde highlight oranını biraz yükseltiyor. Bunu raporda belirtmek gerekir.

In [ ]:
q = """
SELECT
  dimensions IS NULL AS dim_null,
  COUNT(*) AS n,
  COUNTIF(is_highlight) AS n_highlight,
  ROUND(AVG(CAST(is_highlight AS INT64)),4) AS highlight_rate
FROM `the-met-open-access.met_data.mart_met_objects`
WHERE department != 'The Libraries'
GROUP BY dim_null
"""
print(client.query(q).to_dataframe())

## 2. Özellik hazırlama ve train/test bölme

Bu hücre 6 adım yapıyor:

1. **Boyut ayrıştırma:** `"16-5/8 x 9 in."` gibi metinleri cm'ye çevirir.
   Çıktılar: `max_cm` (en uzun kenar), `area_cm2` (yaklaşık alan), `single_dim`
   (tek ölçü varsa 1 — alan bu durumda kaba bir tahmindir).
2. **Lot grubu:** `object_number`'ın ilk iki parçası (ör. `29.100.5` → `29.100`).
   Aynı partiden gelen "kardeş" eserler hem train hem testte olursa model ezber yapar;
   bunu önlemek için bölmeyi lota göre yapıyoruz.
3. **Özellikler:** `department`, `classification`, `culture`, boyutlar.
   `is_on_view` **yok** (sızıntı olmasın).
4. **%80 train / %20 test** (lota göre, `GroupShuffleSplit`).
5. **Eksik boyutları** sadece *train* verisinin medyanıyla doldur (test'e bakmadan).
6. **Baseline:** "Her esere kendi bölümünün highlight oranını ver" — modelin geçmesi gereken basit çıta.

📌 Not: Train highlight oranı %0,49, test %0,64. Lot bazlı bölmede bu fark normal ama
sonuçları yorumlarken akılda tut.

In [ ]:
import re #Metinler içinde belirli desenleri aramak, eşleştirmek, değiştirmek veya metni bölmek için kullanılır.
from sklearn.model_selection import GroupShuffleSplit #kardeş eserleri önce gruplayıp sonra satırları bölüyor.
from sklearn.metrics import average_precision_score #PR-AUC

# ---------- 1) Boyut ayrıştırma ----------
def to_float(tok):#kesirli yazıyı sayıysa çevirir.
    tok = tok.replace("-", " ").strip()          # "16-5/8" -> "16 5/8"
    total = 0.0
    for p in tok.split():#boşuğa göre böler
        if "/" in p:
            a, b = p.split("/")
            total += float(a) / float(b)
        else:
            total += float(p)
    return total

NUM = r"\d+\s+\d+/\d+|\d+-\d+/\d+|\d+/\d+|\d*\.?\d+"#sayı bulan desen

def nums_in(text):#metindeki tüm sayıları listeler
    out = []
    for t in re.findall(NUM, text):
        try:#Döngüdeki sayı için dönüştürme işlemini dener (Hata oluşma riskine karşı bir güvenlik önlemidir).
            out.append(to_float(t))
        except Exception:#hata alırssan durma diğerine geç!
            pass
    return out

def parse_dim(s):
    s = str(s).replace("\\n", " ")
    low = s.lower()
    if "unrecorded" in low:#boyut kayıtsızsa boş döner
        return (np.nan, np.nan, 0)

    nums, factor = None, 1.0 #factor cm ye çevirme çarpanı
    # 1) parantez içinde cm:  (51.4 x 64.6 cm)  /  (H. 31.1 x W. 21.6 cm.)
    m = re.search(r"\(([^)]*?)\s*cm\.?\)", s, flags=re.I)
    if m:
        nums = nums_in(m.group(1))
    # 2) başta cm:  4.9 x 3 x 1.7 cm (...)
    if not nums:
        m = re.search(r"^([^(]*?)\s*cm\b", s, flags=re.I)
        if m:
            nums = nums_in(m.group(1))
    # 3) mm (parantez içi veya düz):  (37 x 32 mm)  /  35mm
    if not nums:
        m = re.search(r"([\d\.\sxX×/;.A-Za-z]*?)\s*mm\b", s)
        if m:
            nums = nums_in(m.group(1))
            factor = 0.1
    # 4) inç:  5 x 7 in.  /  H. 16-5/8, W. 9 inches
    if not nums:
        m = re.search(r"^(.*?)\s*(?:in\b|inch)", s, flags=re.I)
        if m:
            nums = nums_in(m.group(1))#ilk parantezi al
            factor = 2.54

    if not nums:
        return (np.nan, np.nan, 0)
    nums = sorted([n * factor for n in nums], reverse=True)
    if len(nums) == 1:
        return (nums[0], nums[0] ** 2, 1)        # tek ölçü: yaklaşık alan, bayraklı
    return (nums[0], nums[0] * nums[1], 0)

parsed = df["dimensions"].apply(parse_dim)
df["max_cm"]     = parsed.str[0]
df["area_cm2"]   = parsed.str[1]
df["single_dim"] = parsed.str[2]

print("alan parse edilemeyen:", df["area_cm2"].isna().sum(), "/", len(df))
print("tek ölçülü:", int(df["single_dim"].sum()))
print(df.loc[df["area_cm2"].isna(), "dimensions"].sample(10, random_state=1).to_string())

# ---------- 2) Lot grubu ----------
df["lot"] = (df["object_number"].astype(str)
             .str.extract(r"^([^.]+\.[^.]+)")[0]
             .fillna(df["object_number"].astype(str)))

# ---------- 3) Özellikler (is_on_view YOK) ----------
df["y"] = df["is_highlight"].astype(int)
X = df[["department", "classification", "culture", "area_cm2", "max_cm", "single_dim"]].copy()
X["has_area"] = df["area_cm2"].notna().astype(int)
y = df["y"]

# ---------- 4) Lota göre %80/%20 ayır ----------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(X, y, groups=df["lot"]))
X_train, X_test = X.iloc[tr].copy(), X.iloc[te].copy()
y_train, y_test = y.iloc[tr], y.iloc[te]
print("train highlight oranı:", y_train.mean().round(4), "| test:", y_test.mean().round(4))

# ---------- 5) Eksik alanı SADECE train medyanıyla doldur ----------
med_area = X_train.groupby("classification")["area_cm2"].median()
med_max  = X_train.groupby("classification")["max_cm"].median()
glob_area, glob_max = X_train["area_cm2"].median(), X_train["max_cm"].median()
for d in (X_train, X_test):
    d["area_cm2"] = d["area_cm2"].fillna(d["classification"].map(med_area)).fillna(glob_area)
    d["max_cm"]   = d["max_cm"].fillna(d["classification"].map(med_max)).fillna(glob_max)

# ---------- 6) Baseline: department ortalaması (sadece train'den) ----------
dept_rate = y_train.groupby(X_train["department"]).mean()
base = X_test["department"].map(dept_rate).fillna(y_train.mean())
print("Baseline PR-AUC:", average_precision_score(y_test, base).round(4))
print("Rastgele PR-AUC (referans):", y_test.mean().round(4))

## 3. Model: Lojistik regresyon

- Kategorik sütunlar **one-hot** ile sayıya çevriliyor; 50'den az eseri olan kategoriler `other` grubunda toplanıyor.
- Boyutlar çok çarpık olduğu için `log` alınıyor.
- Highlight çok nadir (%0,5) olduğu için `class_weight="balanced"` kullanılıyor.

**Nasıl ölçüyoruz?**
- **PR-AUC:** Nadir sınıflar için ana metriğimiz. Rastgele tahminin PR-AUC'si ≈ highlight oranıdır (0,0064).
- **ROC-AUC:** Genel sıralama başarısı (0,5 = rastgele, 1 = mükemmel).
- **Precision@K:** Modelin en emin olduğu K eserin kaçı gerçekten highlight?

| Model | PR-AUC |
|---|---|
| Rastgele | 0,0064 |
| Baseline (bölüm ortalaması) | 0,0266 |
| Lojistik regresyon | 0,0604 (≈ rastgelenin 9,5 katı) |

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

cat_cols = ["department", "classification", "culture"]
num_cols = ["area_cm2", "max_cm", "single_dim", "has_area"]

# Nadir kategorileri (<50 eser) tek bir "other" grubuna topla (train'den öğren)
for c in cat_cols:
    vc = X_train[c].fillna("missing").value_counts()
    keep = set(vc[vc >= 50].index)
    X_train[c] = X_train[c].fillna("missing").where(X_train[c].fillna("missing").isin(keep), "other")
    X_test[c]  = X_test[c].fillna("missing").where(X_test[c].fillna("missing").isin(keep), "other")

# Alan çok çarpık: log dönüşümü
for d in (X_train, X_test):
    d["area_cm2"] = np.log1p(d["area_cm2"])
    d["max_cm"]   = np.log1p(d["max_cm"])

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=0.5)),
])
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

print("Lojistik Regresyon PR-AUC:", round(average_precision_score(y_test, proba), 4))
print("Lojistik Regresyon ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
print("Baseline PR-AUC:", round(average_precision_score(y_test, base), 4))
print("Rastgele PR-AUC:", round(y_test.mean(), 4))

# Precision@K: en yüksek skorlu K eserin kaçı gerçekten highlight?
order = np.argsort(-proba)
for k in (100, 500, 1000):
    print(f"Precision@{k}:", round(y_test.iloc[order[:k]].mean(), 4))

# Knapsack için skorları sakla
res = df.iloc[te][["object_id", "object_number", "is_on_view", "sergileme_tipi", "area_cm2"]].copy()
res["p_highlight"] = proba
res["y"] = y_test.values

## 4. Olasılıkları gerçek orana çevirme (kalibrasyon)

`class_weight="balanced"` modelin olasılıklarını şişirir (sanki highlight %50 yaygınmış gibi).
Knapsack'te "beklenen highlight sayısı" hesaplayacağımız için olasılıkları gerçek orana geri çekiyoruz.
Kontrol: ortalama kalibre olasılık (0,0063) ≈ gerçek test oranı (0,0064).

Bu hücre `res` tablosunu oluşturur: test eserleri + `p_highlight` (kalibre olasılık) + gerçek etiket `y`.

In [ ]:
# Olasılığı gerçek orana geri çevir (class_weight="balanced" düzeltmesi)
r = y_train.sum() / (len(y_train) - y_train.sum())
p_cal = proba * r / (proba * r + (1 - proba))

res = df.iloc[te][["object_id", "object_number", "is_on_view",
                   "sergileme_tipi", "area_cm2", "single_dim"]].copy()
res["has_area"] = res["area_cm2"].notna().astype(int)
res["p_raw"] = proba
res["p_highlight"] = p_cal
res["y"] = y_test.values

print("Ortalama kalibre olasılık:", round(p_cal.mean(), 4), "| gerçek test oranı:", round(y_test.mean(), 4))
print("Knapsack havuzu (gerçek 2 ölçülü):", int(((res.has_area == 1) & (res.single_dim == 0)).sum()))

## 5. Tek oda knapsack

**Soru:** 50 m²'lik bir odaya, modelin hiç görmediği test eserlerinden hangilerini koyarsak
**en çok highlight** bekleriz?

- Her eser yer kaplar: `alan × boşluk katsayısı`.
- Çeşitlilik için aynı sınıftan en fazla 10 eser.
- **Greedy:** "olasılık / kapladığı alan" oranı en yüksekten başlayarak doldur.
- **LP üst sınırı:** Teorik en iyi değer; hiçbir yöntem bunu geçemez. Greedy'nin buna yakınlığına bakıyoruz.

### 5.1 İlk deneme (eski sürüm)
Bu ilk sürümde `Unknown` sınıfı ve kaba sergileme tipleri vardı. Karşılaştırma için tutuluyor;
**sonraki hücreler bunu kullanmıyor**, atlanabilir.

In [ ]:
# ---------- Knapsack: tek oda — İLK DENEME (sonraki hücreler kullanmaz) ----------
rng = np.random.default_rng(42)

# Havuz: gerçek iki ölçülü, boyutu bilinen eserler (modelin hiç görmediği test verisi)
pool = res[(res.has_area == 1) & (res.single_dim == 0)].copy()
pool = pool[pool.area_cm2 > 0]
pool["classification"] = df.loc[pool.index, "classification"].values

K_BASE = {"Duvar": 2.0, "Vitrin": 3.0, "Zemin": 4.0}   # boşluk katsayısı (varsayım)
ROOM_M2 = 50                                            # oda büyüklüğü (m²)
CAP = ROOM_M2 * 10_000                                  # cm²
MAX_PER_CLASS = 10                                      # çeşitlilik: aynı türden en fazla 10 eser

def solve(pool, scale=1.0, cap=CAP, max_per_class=MAX_PER_CLASS):
    d = pool.copy()
    d["w"] = d.area_cm2 * d.sergileme_tipi.map(K_BASE) * scale   # efektif alan
    d["ratio"] = d.p_highlight / d.w
    d = d.sort_values("ratio", ascending=False)

    # Greedy + çeşitlilik sınırı
    used, cnt, picked = 0.0, {}, []
    for idx, row in d.iterrows():
        c = row.classification
        if used + row.w <= cap and cnt.get(c, 0) < max_per_class:
            picked.append(idx); used += row.w; cnt[c] = cnt.get(c, 0) + 1
    g = d.loc[picked]

    # LP üst sınırı (kesirli knapsack, çeşitlilik sınırı yok => gevşek üst sınır)
    cw = d.w.cumsum()
    full = d[cw <= cap]
    rest = cap - full.w.sum()
    nxt = d[cw > cap].head(1)
    ub = full.p_highlight.sum() + (nxt.p_highlight.iloc[0] * rest / nxt.w.iloc[0] if len(nxt) else 0)
    return g, ub

def random_pick(pool, scale=1.0, cap=CAP):
    d = pool.sample(frac=1, random_state=1).copy()
    d["w"] = d.area_cm2 * d.sergileme_tipi.map(K_BASE) * scale
    return d[d.w.cumsum() <= cap]

print(f"Havuz: {len(pool)} eser | oda: {ROOM_M2} m²\n")
print(f"{'k çarpanı':<10}{'seçilen':>8}{'E[highlight]':>14}{'LP üst sınır':>14}{'gerçek hl':>11}{'rastgele hl':>13}")
for s in (0.5, 1.0, 2.0):
    g, ub = solve(pool, scale=s)
    r = random_pick(pool, scale=s)
    print(f"{s:<10}{len(g):>8}{g.p_highlight.sum():>14.2f}{ub:>14.2f}{int(g.y.sum()):>11}{int(r.y.sum()):>13}")

g, _ = solve(pool, 1.0)
print("\nSeçilen eserlerin sergileme tipi dağılımı:")
print(g.sergileme_tipi.value_counts())
print("\nEn çok seçilen sınıflar:")
print(g.classification.value_counts().head(8))

### 5.2 Yardımcı: En sık 60 sınıf

Sergileme tipi kurallarını (`yeni_tip`) genişletmek için en sık sınıflara baktık.

In [ ]:
print(df["classification"].value_counts().head(60).to_string())

### 5.3 Son sürüm

Değişiklikler: daha geniş anahtar kelime listesi (`yeni_tip`), `Unknown` sınıfı çıkarıldı,
LP üst sınırı artık çeşitlilik kuralını da hesaba katıyor, rastgele seçim 20 denemenin ortalaması.

**Sonuç (k=1):** Greedy 740 eser seçti; içlerinde gerçekte **10 highlight** var.
Rastgele seçim ortalama **0,45** highlight buluyor. Greedy, LP üst sınırının ~%81'ine ulaşıyor.

Bu hücre sonraki bölümlerin kullandığı `pool` (aday havuzu) tablosunu oluşturur.

In [ ]:
# ---------- Knapsack: tek oda — SON SÜRÜM (pool burada oluşur) ----------
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, vstack

# 1) Sergileme tipi: classification anahtar kelimelerinden (varsayım)
def yeni_tip(c):
    c = str(c).lower()
    duvar = ['print', 'drawing', 'photograph', 'painting', 'textile', 'ephemera',
             'cut paper', 'negative', 'stained', 'relief', 'mosaic', 'mural',
             'tapestr', 'shafted weapon']
    vitrin = ['coin', 'jewel', 'miniature', 'gem', 'seal', 'amulet', 'gold', 'silver',
              'enamel', 'ornament', 'ivory', 'bead', 'ring', 'cameo', 'medal', 'plaquette',
              'scarab', 'netsuke', 'sword furniture', 'sword', 'dagger', 'jade', 'lacquer',
              'glass', 'ceramic', 'vase', 'porcelain', 'pottery', 'book', 'codice']
    if any(w in c for w in duvar):  return 'Duvar'
    if any(w in c for w in vitrin): return 'Vitrin'
    return 'Zemin'                                   # heykel, bronz, terracotta, stucco...

# 2) Havuz: gerçek iki ölçülü, boyutu bilinen, türü belli eserler (test verisi)
pool = res[(res.has_area == 1) & (res.single_dim == 0)].copy()
pool = pool[pool.area_cm2 > 0]
pool["classification"] = df.loc[pool.index, "classification"].astype(str).values
pool = pool[pool.classification.str.lower() != "unknown"]
pool["sergileme_tipi"] = pool.classification.apply(yeni_tip)

print("Havuz:", len(pool), "eser")
print(pool.sergileme_tipi.value_counts().to_string(), "\n")
print("En kalabalık 25 sınıf -> tip (göz at):")
print(pool.groupby(["classification", "sergileme_tipi"]).size()
          .sort_values(ascending=False).head(25).to_string(), "\n")

# 3) Parametreler (hepsi varsayım)
K_BASE = {"Duvar": 2.0, "Vitrin": 3.0, "Zemin": 4.0}   # boşluk katsayısı
ROOM_M2 = 50
CAP = ROOM_M2 * 10_000                                  # cm²
MAX_PER_CLASS = 10                                      # çeşitlilik sınırı

# 4) Çeşitlilik sınırlı LP üst sınırı (kesirli çözüm, hiçbir yöntem bunu geçemez)
def lp_bound(d, cap, max_per_class):
    n = len(d)
    classes, inv = np.unique(d.classification.values, return_inverse=True)
    A_cls = csr_matrix((np.ones(n), (inv, np.arange(n))), shape=(len(classes), n))
    A = vstack([csr_matrix(d.w.values.reshape(1, -1)), A_cls])
    b = np.concatenate([[cap], np.full(len(classes), max_per_class)])
    r = linprog(-d.p_highlight.values, A_ub=A, b_ub=b, bounds=(0, 1), method="highs")
    return -r.fun

# 5) Greedy + çeşitlilik sınırı
def solve(pool, scale=1.0):
    d = pool.copy()
    d["w"] = d.area_cm2 * d.sergileme_tipi.map(K_BASE) * scale
    d["ratio"] = d.p_highlight / d.w
    d = d.sort_values("ratio", ascending=False)
    w, cls = d.w.values, d.classification.values
    used, cnt, picked = 0.0, {}, []
    for i in range(len(d)):
        if used + w[i] <= CAP and cnt.get(cls[i], 0) < MAX_PER_CLASS:
            picked.append(i); used += w[i]; cnt[cls[i]] = cnt.get(cls[i], 0) + 1
    return d.iloc[picked], lp_bound(d, CAP, MAX_PER_CLASS)

# 6) Rastgele seçim: 20 denemenin ortalaması
def random_mean(pool, scale=1.0, n=20):
    out = []
    for s in range(n):
        d = pool.sample(frac=1, random_state=s).copy()
        d["w"] = d.area_cm2 * d.sergileme_tipi.map(K_BASE) * scale
        out.append(d[d.w.cumsum() <= CAP].y.sum())
    return float(np.mean(out))

print(f"{'k çarpanı':<10}{'seçilen':>8}{'E[hl]':>8}{'LP sınır':>10}{'greedy/LP':>11}{'gerçek hl':>11}{'rastgele hl':>13}")
for s in (0.5, 1.0, 2.0):
    g, ub = solve(pool, s)
    print(f"{s:<10}{len(g):>8}{g.p_highlight.sum():>8.2f}{ub:>10.2f}"
          f"{g.p_highlight.sum()/ub:>11.0%}{int(g.y.sum()):>11}{random_mean(pool, s):>13.2f}")

g, _ = solve(pool, 1.0)
print("\nSeçilen eserlerin tip dağılımı (k=1):")
print(g.sergileme_tipi.value_counts().to_string())
print("\nEn çok seçilen sınıflar:")
print(g.classification.value_counts().head(8).to_string())

## 6. Çoklu oda knapsack (MILP)

Daha gerçekçi bir senaryo: **3 oda** (Küçük 30 m², Orta 70 m², Büyük 150 m²).

- **Duvar eserleri** duvar uzunluğunu harcar (eser genişliği + 30 cm boşluk; kapılar için 3 m düşülür).
- **Vitrin/Zemin eserleri** zemin alanını harcar; zeminin %40'ı gezinti için boş kalır.
- Her odada her tipten en az 3 eser, aynı sınıftan en fazla 10 eser.
- Bir eser en fazla bir odaya konur.

**MILP** (tamsayılı doğrusal programlama) bu kurallar altında en iyi çözümü bulur. Hız için her
tipten en iyi 1000 aday seçilir. Tüm sayılar **varsayımdır**; hücrenin en üstündeki
`PARAMETRELER` bölümünden değiştirilebilir.

**Karşılaştırma** (3 oda toplamı, gerçek highlight):

| Yöntem | Gerçek highlight |
|---|---|
| MILP | 25 |
| Greedy | 22 |
| "Met sergisi tarzı" (şu an sergilenen eserlerden rastgele) | 11,8 |
| Rastgele | 2,95 |

"Met sergisi tarzı" `is_on_view` bilgisini kullanır. `is_on_view` galeri numarasının
dolu olmasından türetiliyor; boş galeri numarası eserin depoda olduğunu kesin kanıtlamaz.

In [ ]:
# =====================================================================
# Çoklu oda knapsack (MILP)
# Önceki hücrelerden gerekenler: df, pool (Bölüm 5.3'te oluşan son havuz)
# Ayarlanacak her şey en üstteki PARAMETRELER bölümünde (hepsi varsayım).
# =====================================================================
import numpy as np, pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import csr_matrix, vstack, hstack, identity, block_diag

# ---------------- PARAMETRELER (hepsi varsayım) ----------------
ROOMS = {"Küçük": (6, 5), "Orta": (10, 7), "Büyük": (15, 10)}  # (en_m, boy_m)
DOOR_M = 3.0            # kapı boşlukları toplamı (m), duvardan düşülür
WALK = 0.40             # zeminin gezinti için boş kalacak oranı
K_FLOOR = {"Vitrin": 3.0, "Zemin": 4.0}   # zemin eserleri için boşluk katsayısı
WALL_GAP_CM = 30        # duvar eserleri arası boşluk (cm)
MIN_FOOT = 900          # bir eserin en az kapladığı zemin alanı (cm²): etiket+boşluk
MIN_W, MAX_W = 20, 500  # duvar eseri genişliği alt/üst sınırı (cm)
MAX_PER_CLASS = 10      # bir odada aynı sınıftan en fazla eser
MIN_PER_TYPE = 3        # bir odada her sergileme tipinden en az eser
TOP_N = 1000            # aday havuzu: her tipte p ve p/maliyet'e göre en iyi N
TIME_LIMIT = 120        # MILP süre sınırı (sn)
N_RANDOM = 20           # rastgele baseline tekrar sayısı

TYPES = ["Duvar", "Vitrin", "Zemin"]
R = list(ROOMS); nr = len(R)
wall_cap = np.array([(2 * (a + b) - DOOR_M) * 100 for a, b in ROOMS.values()])      # cm#duvar uzunluğu
floor_cap = np.array([a * b * 10_000 * (1 - WALK) for a, b in ROOMS.values()])      # cm²#kullanılabilir zemin

# ---------------- Havuz hazırlığı ----------------
P = pool.copy()
P["max_cm"] = df.loc[P.index, "max_cm"].values
P = P[P.max_cm.notna()].copy()
P["is_wall"] = P.sergileme_tipi.eq("Duvar")
P = P[~P.is_wall | (P.max_cm <= MAX_W)].copy()
P["is_wall"] = P.sergileme_tipi.eq("Duvar")
P["is_on_view"] = P.is_on_view.fillna(False).astype(bool)
P["wall_cost"] = np.where(P.is_wall, np.maximum(P.max_cm, MIN_W) + WALL_GAP_CM, 0.0)
P["floor_cost"] = np.where(P.is_wall, 0.0,
                           np.maximum(P.area_cm2, MIN_FOOT) * P.sergileme_tipi.map(K_FLOOR).fillna(0))
P["cost"] = np.where(P.is_wall, P.wall_cost, P.floor_cost)
P["eff"] = P.p_highlight / P.cost

# Aday havuzu (MILP'i makul boyutta tutmak için)
parts = []
for t, g in P.groupby("sergileme_tipi"):
    parts += [g.nlargest(TOP_N, "p_highlight"), g.nlargest(TOP_N, "eff")]
C = pd.concat(parts)
C = C[~C.index.duplicated()].reset_index(drop=True)
n = len(C)
print(f"Tüm havuz: {len(P)} | MILP aday havuzu: {n}")
print(C.sergileme_tipi.value_counts().to_string(), "\n")

# ---------------- MILP kurulumu ----------------
# Değişken: x[r, i] = i. eser r. odaya konur  (vektörde r*n + i)
cls_codes, cls_names = pd.factorize(C.classification)
ncls = len(cls_names)
S = csr_matrix((np.ones(n), (cls_codes, np.arange(n))), shape=(ncls, n))
tcode = C.sergileme_tipi.map({t: i for i, t in enumerate(TYPES)}).values
T = csr_matrix((np.ones(n), (tcode, np.arange(n))), shape=(3, n))
Wrow = csr_matrix(C.wall_cost.values.reshape(1, -1))
Frow = csr_matrix(C.floor_cost.values.reshape(1, -1))

A_one = hstack([identity(n, format="csr")] * nr, format="csr")        # eser en fazla 1 odada
A_wall = block_diag([Wrow] * nr, format="csr")                         # duvar uzunluğu
A_floor = block_diag([Frow] * nr, format="csr")                        # zemin alanı
A_cls = block_diag([S] * nr, format="csr")                             # sınıf çeşitliliği
A_type = block_diag([T] * nr, format="csr")                            # tip karışımı (alt sınır)

A = vstack([A_one, A_wall, A_floor, A_cls, A_type], format="csr")
lo = np.concatenate([np.full(n, -np.inf), np.full(nr, -np.inf), np.full(nr, -np.inf),
                     np.full(ncls * nr, -np.inf), np.full(3 * nr, MIN_PER_TYPE)])
hi = np.concatenate([np.ones(n), wall_cap, floor_cap,
                     np.full(ncls * nr, MAX_PER_CLASS), np.full(3 * nr, np.inf)])
cons = LinearConstraint(A, lo, hi)
c = -np.tile(C.p_highlight.values, nr)

r_lp = milp(c, constraints=cons, integrality=np.zeros(n * nr), bounds=Bounds(0, 1))
r_mi = milp(c, constraints=cons, integrality=np.ones(n * nr), bounds=Bounds(0, 1),
            options={"time_limit": TIME_LIMIT, "mip_rel_gap": 0.005})

if r_mi.x is None:
    raise SystemExit(f"MILP çözüm bulamadı: {r_mi.message}\n"
                     "Muhtemelen kısıtlar çok sıkı: MIN_PER_TYPE'ı düşür ya da WALK'u azalt.")
print("MILP durumu:", r_mi.message)

x = r_mi.x.reshape(nr, n) > 0.5

# ---------------- Oda bazlı özet ----------------
satirlar, secimler = [], []
for r, ad in enumerate(R):
    s = C[x[r]].copy()
    s["oda"] = ad
    secimler.append(s)
    satirlar.append({
        "Oda": ad,
        "m²": ROOMS[ad][0] * ROOMS[ad][1],
        "Eser": len(s),
        "E[highlight]": round(s.p_highlight.sum(), 2),
        "Gerçek hl": int(s.y.sum()),
        "Duvar": int((s.sergileme_tipi == "Duvar").sum()),
        "Vitrin": int((s.sergileme_tipi == "Vitrin").sum()),
        "Zemin": int((s.sergileme_tipi == "Zemin").sum()),
        "Sınıf sayısı": s.classification.nunique(),
        "Duvar doluluk": f"{s.wall_cost.sum() / wall_cap[r]:.0%}",
        "Zemin doluluk": f"{s.floor_cost.sum() / floor_cap[r]:.0%}",
    })
oda_tablo = pd.DataFrame(satirlar)
secim = pd.concat(secimler, ignore_index=True)
print("\n=== Oda bazlı sonuç (MILP) ===")
print(oda_tablo.to_string(index=False))

# ---------------- Baseline'lar ----------------
def arrs(D):
    return dict(cls=D.classification.values, isw=D.is_wall.values, wc=D.wall_cost.values,
                fc=D.floor_cost.values, p=D.p_highlight.values, y=D.y.values)

def fill(order, a):
    """Eserleri sırayla ilk sığdığı odaya koyar (aynı kısıtlar, MIN_PER_TYPE hariç)."""
    wl, fl = wall_cap.copy(), floor_cap.copy()
    cnt = [dict() for _ in range(nr)]
    ph, yh = 0.0, 0
    for i in order:
        k = a["cls"][i]
        for r in range(nr):
            if cnt[r].get(k, 0) >= MAX_PER_CLASS:
                continue
            if a["isw"][i]:
                if a["wc"][i] > wl[r]: continue
                wl[r] -= a["wc"][i]
            else:
                if a["fc"][i] > fl[r]: continue
                fl[r] -= a["fc"][i]
            cnt[r][k] = cnt[r].get(k, 0) + 1
            ph += a["p"][i]; yh += a["y"][i]
            break
    return ph, yh

aC, aP = arrs(C), arrs(P)

# Greedy (aday havuzunda, p / göreli maliyet)
cf = np.where(aC["isw"], aC["wc"] / wall_cap.mean(), aC["fc"] / floor_cap.mean())
g_p, g_y = fill(np.argsort(-(aC["p"] / cf)), aC)

# Rastgele ve "Met sergisi tarzı" (tüm havuzda, modelden bağımsız)
rng = np.random.default_rng(42)
onview = np.where(P.is_on_view.values)[0]
rs, ms = [], []
for _ in range(N_RANDOM):
    rs.append(fill(rng.permutation(len(P)), aP))
    ms.append(fill(rng.permutation(onview), aP))
rs, ms = np.array(rs).mean(axis=0), np.array(ms).mean(axis=0)

lp_val, mi_val = -r_lp.fun, -r_mi.fun
karsilastirma = pd.DataFrame({
    "Yöntem": ["LP üst sınır", "MILP (optimum)", "Greedy", "Met sergisi tarzı*", "Rastgele*"],
    "E[highlight]": [lp_val, mi_val, g_p, ms[0], rs[0]],
    "Gerçek highlight": [np.nan, secim.y.sum(), g_y, ms[1], rs[1]],
}).round(2)
print("\n=== Yöntem karşılaştırması (toplam, 3 oda) ===")
print(karsilastirma.to_string(index=False))
print(f"\nMILP / LP sınırı: {mi_val / lp_val:.1%}   |   Greedy / MILP: {g_p / mi_val:.1%}")
print("* Met tarzı ve rastgele: modelden bağımsız, sergilenen (veya tüm) eserlerden rastgele doldurma; "
      f"{N_RANDOM} denemenin ortalaması.")
print("\nSeçilen sınıflar (ilk 10):")
print(secim.classification.value_counts().head(10).to_string())

## 7. Duyarlılık analizi

**Soru:** Oda ayarlarını (gezinti payı, sınıf limiti, duvar doluluğu) değiştirince sonuç değişiyor mu?

7 senaryonun hepsinde MILP seçimi, "Met sergisi tarzı" seçimden **1,9–2,6 kat** daha fazla
gerçek highlight buluyor. Yani sonuç tek bir ayara bağlı değil.

In [ ]:
# =====================================================================
# Duyarlılık analizi — Bölüm 6'daki çoklu oda hücresinden SONRA çalıştır
# (önceki hücrenin değişkenlerini kullanır: A, lo, c, C, n, nr, ncls, aP, onview, ROOMS, DOOR_M)
# Soru: oda ayarlarını değiştirince sonuç ne kadar oynuyor?
# =====================================================================
import numpy as np, pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds

# (ad, WALK = zeminin boş kalacak oranı, sınıf limiti, duvarın en fazla dolu olacağı oran)
SENARYOLAR = [
    ("Baz (WALK .40, sınıf 10, duvar %100)", 0.40, 10, 1.00),
    ("Gezinti payı artsın (WALK .50)",        0.50, 10, 1.00),
    ("Gezinti payı azalsın (WALK .30)",       0.30, 10, 1.00),
    ("Sınıf limiti sıkı (5)",                 0.40,  5, 1.00),
    ("Sınıf limiti gevşek (15)",              0.40, 15, 1.00),
    ("Duvar en fazla %75 dolu",               0.40, 10, 0.75),
    ("Hepsi sıkı (WALK .50, sınıf 5, duvar %75)", 0.50, 5, 0.75),
]
SURE = 60        # her senaryo için MILP süre sınırı (sn)
N_MET = 10       # Met tarzı baseline tekrar sayısı

def kapasite(walk, wfill):
    wc = np.array([(2 * (a + b) - DOOR_M) * 100 for a, b in ROOMS.values()]) * wfill
    fc = np.array([a * b * 10_000 for a, b in ROOMS.values()]) * (1 - walk)
    return wc, fc

def fill2(order, a, wc_cap, fc_cap, maxc):
    """Sırayla ilk sığdığı odaya koyar; gerçek highlight sayısını döndürür."""
    wl, fl = wc_cap.copy(), fc_cap.copy()
    cnt = [dict() for _ in range(nr)]
    yh = 0
    for i in order:
        k = a["cls"][i]
        for r in range(nr):
            if cnt[r].get(k, 0) >= maxc:
                continue
            if a["isw"][i]:
                if a["wc"][i] > wl[r]: continue
                wl[r] -= a["wc"][i]
            else:
                if a["fc"][i] > fl[r]: continue
                fl[r] -= a["fc"][i]
            cnt[r][k] = cnt[r].get(k, 0) + 1
            yh += a["y"][i]
            break
    return yh

rng2 = np.random.default_rng(1)
p_arr, y_arr = C.p_highlight.values, C.y.values
satirlar = []
for ad, walk, maxc, wfill in SENARYOLAR:
    wc_cap, fc_cap = kapasite(walk, wfill)
    hi = np.concatenate([np.ones(n), wc_cap, fc_cap,
                         np.full(ncls * nr, maxc), np.full(3 * nr, np.inf)])
    r = milp(c, constraints=LinearConstraint(A, lo, hi), integrality=np.ones(n * nr),
             bounds=Bounds(0, 1), options={"time_limit": SURE, "mip_rel_gap": 0.01})
    met = np.mean([fill2(rng2.permutation(onview), aP, wc_cap, fc_cap, maxc) for _ in range(N_MET)])
    if r.x is None:
        satirlar.append({"Senaryo": ad, "Eser": np.nan, "E[hl]": np.nan,
                         "Gerçek hl": np.nan, "Met tarzı hl": round(met, 1), "Gerçek/Met": np.nan})
        continue
    sel = (r.x.reshape(nr, n) > 0.5).any(axis=0)
    satirlar.append({"Senaryo": ad, "Eser": int(sel.sum()),
                     "E[hl]": round(p_arr[sel].sum(), 1), "Gerçek hl": int(y_arr[sel].sum()),
                     "Met tarzı hl": round(met, 1),
                     "Gerçek/Met": round(y_arr[sel].sum() / met, 2) if met > 0 else np.nan})
    print("tamam:", ad)

duyarlilik = pd.DataFrame(satirlar)
print("\n=== Duyarlılık analizi (3 oda toplamı) ===")
print(duyarlilik.to_string(index=False))
print("\nOkuma: 'Gerçek/Met' tüm senaryolarda 1'in belirgin üstündeyse sonuç ayarlara bağlı değil, sağlam.")

## 8. Power BI için dışa aktarma

İki zip dosyası indirilir:

- `met_powerbi.zip` → knapsack sonuçları (oda tablosu, yöntem karşılaştırması, duyarlılık, seçilen eserler).
  `metrikler.csv` içindeki değerler elle yazılmıştır; model yeniden eğitilirse güncellemeyi unutma.
- `met_ml_tablolar.zip` → model sonuçları (modeller, Precision@K, train/test bölme, en olası 100 eser).
  Bu tablolar değişkenlerden otomatik hesaplanır.

In [ ]:
import zipfile
from google.colab import files

# 1) Model metrikleri (ekranda gördüğün değerler)
metrikler = pd.DataFrame({
    "Metrik": ["Rastgele PR-AUC", "Baseline PR-AUC", "Lojistik Regresyon PR-AUC",
               "ROC-AUC", "Precision@100", "Precision@500", "Precision@1000"],
    "Değer": [0.0064, 0.0266, 0.0604, 0.877, 0.09, 0.128, 0.104]
})

# 2) Seçilen eserler (oda, sınıf, tip, olasılık, gerçek highlight)
secilen = secim[["oda", "object_id", "object_number", "classification",
                 "sergileme_tipi", "p_highlight", "y", "is_on_view"]].rename(
                 columns={"y": "gercek_highlight"})

tablolar = {
    "metrikler.csv": metrikler,
    "oda_tablo.csv": oda_tablo,
    "yontem_karsilastirma.csv": karsilastirma,
    "duyarlilik.csv": duyarlilik,
    "secilen_eserler.csv": secilen,
}

with zipfile.ZipFile("met_powerbi.zip", "w") as z:
    for ad, t in tablolar.items():
        t.to_csv(ad, index=False, encoding="utf-8-sig")   # Türkçe karakter bozulmasın
        z.write(ad)

files.download("met_powerbi.zip")

In [ ]:
import zipfile
from google.colab import files
from sklearn.metrics import average_precision_score, roc_auc_score

n_pos = int(y_test.sum())
rate = y_test.mean()

ml_modeller = pd.DataFrame({
    "Model": ["Rastgele", "Baseline (bölüm ortalaması)", "Lojistik Regresyon"],
    "PR_AUC": [rate, average_precision_score(y_test, base), average_precision_score(y_test, proba)],
    "ROC_AUC": [0.5, roc_auc_score(y_test, base), roc_auc_score(y_test, proba)],
}).round(4)
ml_modeller["Rastgeleye_kat"] = (ml_modeller["PR_AUC"] / rate).round(1)

order = np.argsort(-proba)
satirlar = []
for k in (100, 250, 500, 1000, 2000):
    hit = int(y_test.iloc[order[:k]].sum())
    satirlar.append({"K": k, "Bulunan_highlight": hit,
                     "Precision": round(hit / k, 4),
                     "Recall": round(hit / n_pos, 4),
                     "Rastgele_precision": round(rate, 4),
                     "Rastgeleye_kat": round((hit / k) / rate, 1)})
ml_precision_k = pd.DataFrame(satirlar)

ml_bolme = pd.DataFrame({
    "Küme": ["Train", "Test"],
    "Eser": [len(y_train), len(y_test)],
    "Highlight": [int(y_train.sum()), n_pos],
    "Highlight_orani": [round(y_train.mean(), 4), round(rate, 4)],
})

te_df = df.iloc[te][["object_id", "object_number", "department", "classification", "culture"]].copy()
te_df["p_model"] = proba
te_df["gercek_highlight"] = y_test.values
ml_top100 = te_df.sort_values("p_model", ascending=False).head(100)

print(ml_modeller.to_string(index=False), "\n")
print(ml_precision_k.to_string(index=False), "\n")
print(ml_bolme.to_string(index=False))

tablolar = {"ml_modeller.csv": ml_modeller, "ml_precision_k.csv": ml_precision_k,
            "ml_bolme.csv": ml_bolme, "ml_top100.csv": ml_top100}
with zipfile.ZipFile("met_ml_tablolar.zip", "w") as z:
    for ad, t in tablolar.items():
        t.to_csv(ad, index=False, encoding="utf-8-sig")
        z.write(ad)
files.download("met_ml_tablolar.zip")

## 9. Sonuçlar ve sınırlamalar

**Ne bulduk?**
- Basit bir lojistik regresyon, highlight eserleri rastgeleye göre ~9,5 kat, bölüm ortalamasına göre
  ~2,3 kat daha iyi sıralıyor (PR-AUC 0,060). En emin olduğu 500 eserin %12,8'i highlight
  (rastgele: %0,64).
- Ama mutlak başarı hâlâ düşük: highlight'ı **büyük ölçüde** belirleyen bilgi bu özelliklerde yok
  (ör. sanatçı ünü, tarihsel önem, küratör kararı).
- Model olasılıkları ile oda seçimi yapınca, "şu an sergilenen eserlerden rastgele seçime" göre
  yaklaşık 2 kat daha fazla highlight bulunuyor; bu sonuç farklı oda ayarlarında da korunuyor.

**Sınırlamalar**
- Bu bir **tahmin** çalışmasıdır; özellikler ile highlight arasındaki ilişki **neden-sonuç** göstermez.
- Sergileme tipi, boşluk katsayıları, oda ölçüleri **varsayımdır**.
- Boyut ayrıştırma kurallara dayalı; ~100 bin eserde tek ölçü var (alan kaba tahmin).
- Sadece boyutu olan eserler kullanıldı; bu filtre highlight oranını değiştiriyor (Bölüm 1.1).
- `is_on_view` galeri numarası doluluğundan türetiliyor; depoda olduğunu kanıtlamaz.
- Sonuçlar tek bir train/test bölmesine dayanıyor (`random_state=42`).

## 10. Kısa sözlük

| Terim | Basit anlamı |
|---|---|
| Highlight | Met'in öne çıkardığı önemli eserler (`is_highlight`). |
| Sızıntı (leakage) | Modele, gerçek hayatta tahmin anında bilinmeyecek bir ipucu vermek. |
| Train / Test | Modelin öğrendiği veri / hiç görmediği, sınav için ayrılan veri. |
| Baseline | Modelin geçmesi gereken basit karşılaştırma yöntemi. |
| PR-AUC | Nadir sınıfta modelin "doğru olanları üste koyma" başarısı. |
| ROC-AUC | Genel sıralama başarısı (0,5 rastgele). |
| Precision@K | En üstteki K tahminin kaçının doğru olduğu. |
| Kalibrasyon | Modelin olasılıklarını gerçek oranlarla uyumlu hâle getirmek. |
| Knapsack | Sınırlı kapasiteye en değerli eşyaları sığdırma problemi. |
| Greedy | "En verimli olandan başla" diye ilerleyen hızlı yöntem. |
| LP üst sınırı | Eserlerin "yarısı" bile seçilebilseydi ulaşılacak en iyi değer; tavan. |
| MILP | Eserleri tam seçen (0/1) ve en iyi çözümü bulan optimizasyon yöntemi. |